# Vision Practice 03. DETR 코드 학습 — 시험 대비 실습본

- 원본: `vision/03_DETR.ipynb`
- 핵심 빈칸 수: **4개**
- `## 정답 입력` 셀을 직접 구현한 뒤 과목별 정답·해설지와 비교하세요.
- API key, 외부 서버 주소, 대용량 데이터 경로는 자신의 환경에 맞게 설정하세요.


# **DETR 실습 노트북 (PyTorch) — 추론 파이프라인부터 Attention 시각화까지**

**DETR(DEtection TRansformer)** 기반 객체 탐지 실습을 목표로 합니다.  
`Backbone(CNN) → Transformer(Encoder/Decoder) → Object Query 기반 예측` 흐름을 따라가며,
**출력(`pred_logits`, `pred_boxes`)이 어떤 의미인지**와 **attention이 어디를 보고 있는지**를 직접 확인합니다.

## 학습 목표
- DETR의 **Backbone / Transformer(Encoder·Decoder) / Object Query** 흐름을 코드에서 찾을 수 있다.
- 모델 출력인 `pred_logits`, `pred_boxes`의 **shape와 의미**를 설명할 수 있다.  
  - `pred_logits`: (B, num_queries, num_classes + 1)에서 **+1이 no-object**임을 이해한다.
  - `pred_boxes`: (B, num_queries, 4)가 **정규화 박스(`cx, cy, w, h`)**임을 이해한다.
- 간단한 COCO 샘플 이미지로 **추론 → 신뢰도 필터링 → 박스 복원 → 시각화**까지 한 번에 실행할 수 있다.
- Forward hook으로 캡처한 **Encoder self-attention / Decoder cross-attention**을 시각화하고 해석할 수 있다.

---

### 구성 개요

1. 환경 설정(패키지 로드, 추론 모드 설정)
2. COCO 클래스/전처리/후처리 준비
3. DETR 사전학습 모델 로드 및 구조 확인
4. 한 장 이미지 추론 + 결과 필터링 + 박스 스케일 복원
5. 탐지 결과 시각화
6. 내부 텐서 추출(forward hook) 및 특징맵/attention 확인
7. Encoder self-attention 2D 복원 및 시각화
8. Object Query 기반 Decoder cross-attention 인터랙티브 시각화


---
## 0) Setup (환경 준비)

- **목적:** 실습에 필요한 패키지를 로드하고, 추론/시각화에 필요한 기본 설정을 준비합니다.
- **관찰 포인트:**
  - 시각화용 패키지(`matplotlib`, `ipywidgets`)와 모델 실행용 패키지(`torch`, `torchvision`)가 함께 사용됨
  - 노트북 출력 품질을 위해 `InlineBackend.figure_format='retina'`가 설정됨
  - `torch.set_grad_enabled(False)`로 **추론 모드**에서 불필요한 gradient 계산을 꺼 속도/메모리를 절약함

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(pkg_name: str, import_name=None):
    """
    패키지 설치 여부를 확인하고, 설치되어 있지 않으면 설치합니다.
    """
    name = import_name or pkg_name
    # 패키지가 설치되어 있는지 확인
    if importlib.util.find_spec(name) is None:
        print(f"[install] {pkg_name} 라이브러리를 설치 중입니다... (import name: {name})")
        try:
            # -q 옵션을 추가하여 설치 과정을 간결하게 유지할 수 있습니다.
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_name])
            print(f"[success] {pkg_name} 설치 완료.")
        except subprocess.CalledProcessError as e:
            print(f"[error] {pkg_name} 설치 실패: {e}")
    else:
        print(f"[ok] {pkg_name} 이미 설치되어 있습니다.")

# 설치가 필요한 패키지 리스트 (패키지명, 임포트명)
# 임포트명이 패키지명과 다른 경우 튜플로 지정합니다.
packages = [
    ("requests", "requests"),
    ("Pillow", "PIL"),        # 설치는 Pillow, 임포트는 PIL
    ("ipywidgets", "ipywidgets"),
    ("torchinfo", "torchinfo")
]

# 루프를 돌며 확인 및 설치
for pkg, imp in packages:
    ensure_package(pkg, imp)

In [ ]:
# [DETR 실습] 라이브러리/도구 임포트
# - 논문 구현 흐름: (1) 이미지 전처리 → (2) DETR 추론 → (3) 로짓/박스 후처리 → (4) 시각화
# - 여기서는 공식 DETR 데모(ResNet-50 backbone) 실행에 필요한 패키지를 불러옵니다.

import math  # 수학 유틸(예: 스케일링/좌표 변환)에 사용

from PIL import Image  # PIL 이미지 로딩/처리
import requests  # COCO 샘플 이미지 다운로드
import matplotlib.pyplot as plt  # 결과(박스/어텐션) 시각화
%config InlineBackend.figure_format = 'retina'  # 노트북에서 고해상도 렌더링

import ipywidgets as widgets  # 인터랙티브 위젯(UI) 생성
from IPython.display import display, clear_output  # 위젯/출력 업데이트

import torch  # PyTorch 텐서/모델 실행
from torch import nn  # (일부 셀에서) 모듈 타입 접근용
from torchvision.models import resnet50  # (참고용) backbone 예시
import torchvision.transforms as T  # 입력 전처리(Resize/Normalize 등)
torch.set_grad_enabled(False);  # 추론/시각화 목적이므로 gradient 계산 비활성화(속도/메모리 절약)


---
## 1) COCO 클래스 이름 테이블

- **목적:** DETR(COCO pretrained)의 클래스 인덱스를 문자열로 매핑합니다.
- **관찰 포인트**
  - 모델의 `pred_logits`는 **클래스 인덱스**로 출력되므로, 이를 **라벨 문자열**로 바꾸기 위해 `CLASSES`가 필요함
  - `N/A`는 COCO에서 사용하지 않는 슬롯이거나, 모델 구조상 남아있는 자리일 수 있음


In [ ]:
# [DETR] COCO 클래스 이름 테이블
# - DETR-ResNet50 (COCO pretrained)의 클래스 인덱스를 사람이 읽을 수 있는 문자열로 매핑합니다.
# - 마지막 'N/A'들은 COCO에서 사용하지 않는 슬롯을 의미합니다.

# COCO classes
CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A',
    'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]

print(f"Total number of classes: {len(CLASSES)}")
valid_classes = [c for c in CLASSES if c != 'N/A']
print(f"Effective number of classes: {len(valid_classes)}")


# colors for visualization
COLORS = [[0.000, 0.447, 0.741], [0.850, 0.325, 0.098], [0.929, 0.694, 0.125],
          [0.494, 0.184, 0.556], [0.466, 0.674, 0.188], [0.301, 0.745, 0.933]]


---
## 2) 입력 전처리 및 박스 좌표 후처리

- **목적:** DETR 입력 규격에 맞게 이미지를 전처리하고, 모델이 예측한 박스를 원본 이미지 좌표계로 되돌리는 함수를 준비합니다.
- **관찰 포인트**
  - `T.Compose([...])`는 **Resize/Normalize** 등 입력 변환을 순차적으로 적용함
  - DETR의 박스는 보통 정규화 좌표(`cx, cy, w, h`) 형태이므로, 시각화를 위해 **픽셀 좌표**로 스케일 복원이 필요함
  - 후처리 함수는 (정규화 → 픽셀) 변환과 `(cx,cy,w,h) → (x1,y1,x2,y2)` 변환 역할을 담당함


In [ ]:
# ## 정답 입력
# Drill 1: 2) 입력 전처리 및 박스 좌표 후처리
# 원본 Cell 007의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 3) 탐지 결과 시각화 함수

- **목적:** 모델의 예측 결과(클래스 확률, 박스)를 이미지 위에 그려서 직관적으로 확인합니다.
- **관찰 포인트**
  - 입력으로 `prob`(클래스 확률)과 `boxes`(좌표)를 받아, **선택된 예측만** 표시하도록 설계됨
  - box를 그릴 때는 (x1,y1,x2,y2) 형태가 가장 다루기 쉬움
  - 시각화 품질이 실습 이해도에 크게 영향을 주므로, 폰트/두께/라벨 표시 방식에 주목


In [ ]:
# [DETR] 결과 시각화 함수
# - DETR은 '고정 개수의 object queries'에 대해 클래스 확률과 박스를 예측합니다.
# - 아래 함수는 confidence가 높은 예측들만 골라, 원본 이미지 위에 박스와 클래스/점수를 그립니다.

def plot_results(pil_img, prob, boxes):  # (입력 이미지, 클래스 확률, 박스 좌표)를 받아 시각화
    plt.figure(figsize=(16,10))  # 출력 크기 설정
    plt.imshow(pil_img)  # 이미지 표시
    ax = plt.gca()  # 현재 축(Axes) 얻기
    colors = COLORS * 100  # 박스 색상 반복 사용
    for p, (xmin, ymin, xmax, ymax), c in zip(prob, boxes.tolist(), colors):  # 각 예측(확률 p, 박스, 색상)에 대해 반복
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,  # (xmin,ymin,w,h) 사각형 패치 추가
                                   fill=False, color=c, linewidth=3))
        cl = p.argmax()  # 가장 높은 확률의 클래스 인덱스
        text = f'{CLASSES[cl]}: {p[cl]:0.2f}'  # 라벨 텍스트 구성: 'class: score'
        ax.text(xmin, ymin, text, fontsize=15,  # 박스 좌상단에 라벨 텍스트 표시
                bbox=dict(facecolor='yellow', alpha=0.5))
    plt.axis('off')  # 축 표시 제거
    plt.show()  # 그림 렌더링


---
## 4) 사전학습 DETR 모델 로드

- **목적:** `torch.hub`를 이용해 COCO로 사전학습된 DETR-ResNet50 모델을 로드합니다.
- **관찰 포인트**
  - `pretrained=True`로 로드된 모델은 **바로 추론**이 가능함
  - DETR은 **NMS 없이** set prediction 형태로 결과를 내도록 학습된 모델임
  - 이후 셀에서 출력되는 `pred_logits`, `pred_boxes`의 shape가 `num_queries`에 의해 결정됨


In [ ]:
# [DETR] 사전학습 모델 로드
# - 공식 구현(facebookresearch/detr)의 pretrained 모델을 torch.hub로 로드합니다.
# - backbone=ResNet-50, transformer encoder-decoder + object queries 구조.

model = torch.hub.load('facebookresearch/detr', 'detr_resnet50', pretrained=True)  # 가중치 포함 pretrained DETR-ResNet50 로드
model.eval();  # Dropout/BN 등을 추론 모드로 전환

---
## 5) 모델 구조 출력

- **목적:** DETR 모델의 큰 구성(backbone, transformer, prediction head)을 출력하여 구조를 한 번 확인합니다.
- **관찰 포인트**
  - backbone(CNN)이 spatial feature map을 만들고, transformer가 query 기반으로 객체 표현을 형성함
  - 마지막 head가 `class logits`와 `box`를 각각 예측함
  - 출력 구조를 보면 **어텐션을 어디서 뽑을지(hook 지점)**를 설계할 수 있음


In [ ]:

# ============================================================
# DETR 전체 모델 구조 출력 (모델 로드 직후)
# ------------------------------------------------------------
# torch.hub.load 로 불러온 detr_resnet50 모델의
# PyTorch Module 계층 구조를 출력합니다.
#
# 논문 관점 대응:
# - backbone  → CNN feature extractor (ResNet-50)
# - transformer → Encoder / Decoder (object queries)
# - class_embed / bbox_embed → prediction heads
# ============================================================

print("=== DETR Model Architecture ===")
print(model)


---
## 6) COCO 샘플 이미지 준비

- **목적:** 실습에 사용할 입력 이미지를 다운로드/로딩하여 PIL 이미지로 준비합니다.
- **관찰 포인트**
  - URL에서 이미지를 받아오는 과정(네트워크/권한/방화벽)에서 실패할 수 있음
  - 이후 전처리(transform) 단계에서 PIL → tensor로 변환되며 shape가 `(C,H,W)`로 바뀜


In [ ]:
# [DETR] COCO 샘플 이미지 준비
# - 논문/데모에서 자주 쓰는 COCO val 이미지(고양이/강아지 예시)를 다운로드합니다.

url = 'http://images.cocodataset.org/val2017/000000039769.jpg'  # COCO val 이미지 URL
im = Image.open(requests.get(url, stream=True).raw)  # 다운로드한 바이트 스트림을 PIL 이미지로 디코딩


---
## 7) 한 장 이미지 추론 및 필터링

- **목적:** 이미지 1장을 DETR에 입력하여 `pred_logits`, `pred_boxes`를 얻고, 신뢰도 기준으로 예측을 선택합니다.
- **관찰 포인트**
  - `outputs`는 dict 형태이며 대표 키는 `pred_logits`, `pred_boxes`
  - `pred_logits` shape: `(B, num_queries, num_classes+1)` (**+1은 no-object**)
  - `pred_boxes` shape: `(B, num_queries, 4)` (정규화된 박스)
  - `keep`(마스크/인덱스)로 threshold 이상 예측만 남기면, 시각화가 훨씬 명확해짐


In [ ]:
# [DETR] 한 장 이미지 추론 + 신뢰도 필터링 + 박스 스케일 복원
# - outputs['pred_logits']: (1, num_queries, num_classes+1) 로짓. 마지막 '+1'은 'no-object' 클래스.
# - outputs['pred_boxes']: (1, num_queries, 4) 박스 (cx,cy,w,h) 정규화 좌표.
# - 논문처럼 모든 쿼리가 박스를 내지만, 여기서는 시각화를 위해 높은 confidence만 남깁니다.

# mean-std normalize the input image (batch-size: 1)
img = transform(im).unsqueeze(0)  # 전처리 후 배치 차원 추가: (1,3,H,W)

from torchinfo import summary
model_summary = summary(model,
        input_data=img,
        col_names=["input_size", "output_size", "num_params"],
        depth=3)
print(model_summary)

# propagate through the model
outputs = model(img)  # DETR forward → 로짓/박스 예측

# print keys of outputs
for key in outputs.keys():
    # value를 가져와서 shape 출력
    shape = outputs[key].shape
    print(f"Key : {key}")
    print(f"Shape   : {shape}")
    print("-" * 30)  # 구분선 추가

# keep only predictions with 0.9+ confidence
probas = outputs['pred_logits'].softmax(-1)[0, :, :-1]  # 쿼리별 클래스 확률 계산(no-object 제외): shape=(num_queries,num_classes)
keep = probas.max(-1).values > 0.9  # 각 쿼리의 max class score가 임계값을 넘는 것만 선택
print(f"keep Shape : {keep.shape}")


# convert boxes from [0; 1] to image scales
bboxes_scaled = rescale_bboxes(outputs['pred_boxes'][0, keep], im.size)  # 선택된 박스를 픽셀 좌표로 변환
print(f"bboxes_scaled Shape : {bboxes_scaled.shape}")


---
## 8) 탐지 결과 그리기

- **목적:** 필터링된 예측만을 이용해 bounding box와 class label을 이미지 위에 표시합니다.
- **관찰 포인트**
  - `plot_results(im, probas[keep], bboxes_scaled)`처럼 **필터링 결과만 전달**하는 흐름이 중요함
  - threshold를 조절하면 박스 개수/정확도가 달라지므로 실습 중 직접 바꿔보면 좋음


In [ ]:
# [DETR] 탐지 결과 그리기
# - keep 마스크로 선택된 쿼리들의 (클래스 확률, 박스)를 plot_results로 시각화합니다.

plot_results(im, probas[keep], bboxes_scaled)  # 선택된 예측만 표시


---
## 9) 내부 텐서 추출을 위한 forward hook

- **목적:** DETR의 중간 특징맵(conv features)과 encoder/decoder attention weight를 캡처하기 위해 hook을 등록합니다.
- **관찰 포인트**
  - `register_forward_hook`은 **forward 중간 출력**을 저장하는 전형적인 디버깅/분석 기법
  - 어떤 레이어에 hook을 걸었는지에 따라 얻는 텐서의 의미가 크게 달라짐
  - 이후 attention 시각화는 이 셀에서 저장한 텐서(conv/enc_attn/dec_attn)에 의존함


In [ ]:
# ## 정답 입력
# Drill 2: 9) 내부 텐서 추출을 위한 forward hook
# 원본 Cell 021의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 10) Decoder cross-attention 시각화

- **목적:** Decoder의 cross attention map을 시각화 합니다.
- **관찰 포인트**
  - 입력 이미지 해상도에 비해 feature map은 다운샘플링되어 `(h, w)`가 작아짐
  - 이 `(h, w)`는 attention을 2D로 복원할 때 기준이 되며, 스케일 팩터(factor)와도 연결됨
  - decoder attention 맵 확인 : 물체 외곽이 부각됨


In [ ]:
# ## 정답 입력
# Drill 3: 10) Decoder cross-attention 시각화
# 원본 Cell 023의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 11) Encoder self-attention과 feature map 확인

- **목적:** encoder self-attention이 어떤 형태로 저장되어 있는지(차원/레이어) 확인하고, backbone feature와의 대응을 이해합니다.
- **관찰 포인트**
  - self-attention은 토큰 간 관계를 나타내므로 보통 `(num_heads, HW, HW)` 또는 유사 형태로 나타남
  - head별로 주목하는 영역이 달라서, 특정 head만 시각화해도 해석이 달라질 수 있음


In [ ]:
# [DETR] Encoder self-attention과 backbone feature map 확인
# - enc_attn_weights[0]: (num_heads, HW, HW) 형태(구현에 따라 batch/head 차원 위치가 다를 수 있음).
# - f_map.tensors: backbone 출력(마스크 포함 NestedTensor)에서 실제 feature tensor.

# output of the CNN
f_map = conv_features['0']  # NestedTensor 딕셔너리에서 마지막 stage feature 선택('0' 키)
print("Feature map:            ", f_map.tensors.shape)  # feature map tensor shape 출력: (B,C,H,W)
print("Encoder attention:      ", enc_attn_weights[0].shape)  # encoder 마지막 layer self-attention weight shape 출력

---
## 12) Encoder self-attention을 2D 공간으로 복원

- **목적:** 토큰 기반 attention을 `(h, w)` 격자 형태의 2D heatmap으로 변환하는 준비를 합니다.
- **관찰 포인트**
  - attention의 축(쿼리/키 방향)을 어떤 기준으로 잡는지에 따라 heatmap 의미가 달라짐
  - `reshape`/`view`를 할 때 `(h*w)`가 정확히 맞아야 함 (shape mismatch가 흔한 실수 포인트)


In [ ]:
# [DETR] Encoder self-attention을 2D 공간으로 복원
# - Transformer는 (H*W) 토큰으로 flatten된 sequence를 보지만,
#   시각화를 위해 attention을 (H,W,H,W) 형태로 reshape합니다.

# get the HxW shape of the feature maps of the CNN
shape = f_map.tensors.shape[-2:]  # feature map의 (H,W) 추출
# and reshape the self-attention to a more interpretable shape
sattn = enc_attn_weights[0].reshape(shape + shape)  # (HW,HW) → (H,W,H,W)로 재배열하여 픽셀 위치 간 attention으로 해석
print("Reshaped self-attention:", sattn.shape)  # reshape 결과 shape 확인 -> [H_key, W_key, H_query, W_query] 의미로 해석


---
## 13) Encoder self-attention 시각화

- **목적:** 특정 기준점(픽셀/격자 위치)에서 encoder가 이미지의 어느 위치를 참고하는지 시각화합니다.
- **관찰 포인트**
  - `fact`(downsample factor)를 이용해 원본 좌표 ↔ feature grid 좌표를 매핑함
  - 클릭/지정한 위치에 따라 attention 분포가 달라지며, 전역/국소 문맥을 어떻게 쓰는지 관찰 가능
  - head를 바꿔가며 보면 “관계(relationship)”를 보는 head가 따로 존재함을 느낄 수 있음


In [ ]:
# [DETR] Encoder self-attention 시각화(특정 기준점에서 어디를 보는지)
# - 논문 관점: encoder는 이미지의 모든 위치(HW 토큰) 사이의 self-attention으로 전역 문맥을 섞습니다.
# - 아래는 몇 개의 기준 (y,x) 위치를 고르고, 그 토큰이 다른 위치들을 얼마나 참고하는지 heatmap으로 표시합니다.

# downsampling factor for the CNN, is 32 for DETR and 16 for DETR DC5
fact = 32  # backbone downsample factor(ResNet50 C5 기준 32)

# let's select 4 reference points for visualization
idxs = [(200, 200), (280, 400), (200, 600), (440, 800),]  # 원본 이미지 좌표계에서 기준점들 선택(y,x)

# here we create the canvas
fig = plt.figure(constrained_layout=True, figsize=(25 * 0.7, 8.5 * 0.7))  # 멀티 패널 캔버스 생성
# and we add one plot per reference point
gs = fig.add_gridspec(2, 4)  # 그리드 레이아웃 정의(2x4)
axs = [  # 각 기준점에 대응하는 subplot 생성
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[1, 0]),
    fig.add_subplot(gs[0, -1]),
    fig.add_subplot(gs[1, -1]),
]

# for each one of the reference points, let's plot the self-attention
# for that point
for idx_o, ax in zip(idxs, axs):  # 각 기준점에 대해: feature-map 좌표로 변환 후 attention slice 시각화
    idx = (idx_o[0] // fact, idx_o[1] // fact)  # (y,x) 픽셀 → (y//fact, x//fact) 토큰 좌표
    ax.imshow(sattn[..., idx[0], idx[1]], cmap='cividis', interpolation='nearest')  # 해당 토큰의 attention 분포를 2D heatmap으로 표시
    ax.axis('off')
    ax.set_title(f'self-attention{idx_o}')

# and now let's add the central image, with the reference points as red circles
fcenter_ax = fig.add_subplot(gs[:, 1:-1])  # 가운데에 원본 이미지를 크게 표시
fcenter_ax.imshow(im)
for (y, x) in idxs:
    scale = im.height / img.shape[-2]
    x = ((x // fact) + 0.5) * fact
    y = ((y // fact) + 0.5) * fact
    fcenter_ax.add_patch(plt.Circle((x * scale, y * scale), fact // 2, color='r'))  # 선택한 기준점 위치를 빨간 원으로 표시
    fcenter_ax.axis('off')


---
## 14) AttentionVisualizer (Encoder Self-attention) 클래스

- **목적:** 이미지 내 특정 지점(Reference Point)을 기준으로 **Encoder가 전역적인 문맥(Global Context)을 어떻게 수집하는지** 확인하는 인터랙티브 도구를 정의합니다.
- **관찰 포인트**
  - **Encoder Self-attention의 역할:** 이미지의 한 픽셀(Query)이 '자기 자신을 정의하기 위해' 이미지 전체의 다른 픽셀들(Keys)을 얼마나 참고하는지 나타냄
  - **인터랙티브 시각화:** 슬라이더를 통해 이미지 위의 **빨간 점(Query 좌표)**을 옮기면, 해당 위치의 토큰이 바라보는 **Attention Map**이 실시간으로 업데이트됨.
  - **객체 실루엣 인지:** 빨간 점을 객체(예: 고양이) 위에 두면 어텐션이 해당 객체의 전체적인 형태를 따라 활성화되는 것을 볼 수 있음. 이는 인코더가 이미 객체와 배경을 분리하여 인지하고 있음을 보여줌.
  - **전역 수용장(Global Receptive Field):** CNN과 달리 멀리 떨어진 픽셀 간에도 강한 상관관계를 맺는 것을 확인하며, Transformer가 어떻게 이미지 전체의 문맥을 한 번에 섞는지 이해하는 것이 핵심 목표임.


In [ ]:
# ## 정답 입력
# Drill 4: 14) AttentionVisualizer (Encoder Self-attention) 클래스
# 원본 Cell 031의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 15) Encoder Self-attention 인터랙티브 시각화 실행

- **목적:** 위에서 정의한 `AttentionVisualizer`를 실행하여, 이미지 내 특정 지점(Reference Point)을 기준으로 인코더가 수집한 전역 문맥(Self-attention)을 확인합니다.
- **관찰 포인트**
  - **좌표 이동(X, Y Slider):** 슬라이더를 움직여 빨간 점의 위치를 바꾸면, 해당 픽셀이 이미지의 어떤 영역들과 '관계'를 맺고 있는지 히트맵이 실시간으로 변합니다.
  - **전역 문맥 확인:** 특정 객체(예: 고양이)의 일부분만 찍어도 인코더가 해당 객체의 전체 형태를 파악하고 있는지(히트맵이 객체 실루엣을 따라 활성화되는지) 확인해 보세요.
  - **Attention 방향성(Checkbox):** 'Direction of self attention' 옵션을 통해 '내가 남을 보는 경우'와 '남이 나를 보는 경우'의 차이를 관찰할 수 있습니다.

In [ ]:
# [DETR] Encoder self-attention을 이용한 인터랙티브 시각화 도구 실행
# - AttentionVisualizer는 사용자가 선택한 (1) 이미지 상의 특정 좌표(Red Dot)와 (2) 해당 위치 토큰의 encoder self-attention 맵을 보여줍니다.
# - 논문 관점: 인코더는 이미지 전체를 훑으며 각 픽셀 토큰이 서로의 정보를 참조하게 만듭니다.
# - 이를 통해 모델이 객체의 경계나 배경과의 관계를 어떻게 전역적으로 이해하는지 직관적으로 확인할 수 있습니다.

w = AttentionVisualizer(model, transform)  # 위젯 생성 및 화면에 표시
w.run()
